# Latihan 8: Pseudo Labelling

**Nama:** daffa Abdiantoro  
**NIM:** A11.2024.15898  
**Jurusan:** Teknik Informatika  

---

### Keterangan Latihan:
Latihan ini bertujuan untuk mempraktikkan teknik **Pseudo Labelling** yang merupakan salah satu metode pembelajaran semi-supervised (*semi-supervised learning*):
1. Membuat label prediksi sementara (*pseudo-labels*) untuk data pengujian (*test set*) menggunakan model yang dilatih pada data latih yang ada.
2. Memilih porsi data berlabel pseudo tersebut untuk digabungkan kembali ke data latih guna memperbesar data latih.
3. Melatih ulang model akhir menggunakan data latih yang sudah diperbesar.

Dataset yang digunakan: `Train.csv` dan `Test.csv` (BigMart Sales Prediction dataset).

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import cross_val_score

## 2. Load & Preprocess Dataset
Kita memuat `Train.csv` dan `Test.csv`, lalu menerapkan data preprocessing seperti imputasi nilai kosong dan encoding label.

In [ ]:
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')

# Imputasi Item_Weight menggunakan rata-rata
train['Item_Weight'].fillna((train['Item_Weight'].mean()), inplace=True)
test['Item_Weight'].fillna((test['Item_Weight'].mean()), inplace=True)

# Menyederhanakan kategori fat content menjadi dua kelas utama
for df_temp in [train, test]:
    df_temp['Item_Fat_Content'] = df_temp['Item_Fat_Content'].replace(['low fat','LF'], ['Low Fat','Low Fat'])
    df_temp['Item_Fat_Content'] = df_temp['Item_Fat_Content'].replace(['reg'], ['Regular'])
    df_temp['Outlet_Establishment_Year'] = 2013 - df_temp['Outlet_Establishment_Year']
    df_temp['Outlet_Size'].fillna('Small', inplace=True)

# Label Encoding untuk kolom kategorikal
cat_cols = ['Outlet_Size','Outlet_Location_Type','Outlet_Type','Item_Fat_Content']
test['Item_Outlet_Sales'] = 0
combi = pd.concat([train, test], ignore_index=True)
le = LabelEncoder()
for col in cat_cols:
    combi[col] = le.fit_transform(combi[col].astype(str))

train = combi[:train.shape[0]].copy()
test = combi[train.shape[0]:].copy()
test.drop(columns=['Item_Outlet_Sales'], inplace=True)

# Memisahkan fitur dan target
features = [col for col in train.columns if col not in ['Item_Identifier', 'Outlet_Identifier', 'Item_Type', 'Item_Outlet_Sales']]
target = 'Item_Outlet_Sales'

X_train = train[features]
y_train = train[target]
X_test = test[features]

print("Fitur yang digunakan:", features)
print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test:", X_test.shape)

## 3. Implementasi PseudoLabeler Wrapper Class
Kita membungkus estimator scikit-learn/xgboost agar mendukung proses pseudo labelling semi-supervised secara otomatis.

In [ ]:
class PseudoLabeler(BaseEstimator, RegressorMixin):
    def __init__(self, model, unlabled_data, features, target, sample_rate=0.2, seed=42):
        assert sample_rate <= 1.0, 'sample_rate harus berada di antara 0.0 dan 1.0.'
        self.sample_rate = sample_rate
        self.seed = seed
        self.model = model
        self.model.seed = seed
        self.unlabled_data = unlabled_data
        self.features = features
        self.target = target
        
    def get_params(self, deep=True):
        return {
            "sample_rate": self.sample_rate,
            "seed": self.seed,
            "model": self.model,
            "unlabled_data": self.unlabled_data,
            "features": self.features,
            "target": self.target
        }

    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

    def fit(self, X, y):
        augmented_train = self.__create_augmented_train(X, y)
        self.model.fit(
            augmented_train[self.features],
            augmented_train[self.target]
        )
        return self

    def __create_augmented_train(self, X, y):
        num_of_samples = int(len(self.unlabled_data) * self.sample_rate)
        
        # 1. Train model pada data berlabel awal
        self.model.fit(X, y)
        
        # 2. Predict target pada data test tak berlabel
        pseudo_labels = self.model.predict(self.unlabled_data[self.features])
        
        # 3. Gabungkan data baru ke data latih
        pseudo_data = self.unlabled_data.copy(deep=True)
        pseudo_data[self.target] = pseudo_labels
        
        sampled_pseudo_data = pseudo_data.sample(n=num_of_samples, random_state=self.seed)
        temp_train = pd.concat([X, y], axis=1)
        augmented_train = pd.concat([sampled_pseudo_data, temp_train], ignore_index=True)

        return shuffle(augmented_train, random_state=self.seed)
        
    def predict(self, X):
        return self.model.predict(X)

## 4. Evaluasi & Perbandingan Model
Kita membandingkan performa XGBoost Regressor biasa dengan model XGBoost Regressor yang menggunakan Pseudo Labelling melalui cross validation (8 folds).

In [ ]:
num_folds = 8

# 1. Model Baseline XGBoost
baseline_model = XGBRegressor(nthread=1, random_state=42)
scores_base = cross_val_score(baseline_model, X_train, y_train, cv=num_folds, scoring='neg_mean_squared_error')
rmse_base = np.sqrt(scores_base.mean() * -1)
print(f"XGBRegressor Baseline - CV RMSE: {rmse_base:.4f} (+/- {scores_base.std() * 2:.4f})")

# 2. Model PseudoLabeler
pseudo_model = PseudoLabeler(
    XGBRegressor(nthread=1, random_state=42),
    test,
    features,
    target,
    sample_rate=0.3,
    seed=42
)
scores_pseudo = cross_val_score(pseudo_model, X_train, y_train, cv=num_folds, scoring='neg_mean_squared_error')
rmse_pseudo = np.sqrt(scores_pseudo.mean() * -1)
print(f"PseudoLabeler Model - CV RMSE: {rmse_pseudo:.4f} (+/- {scores_pseudo.std() * 2:.4f})")

## 5. Kesimpulan
- Pseudo Labelling memanfaatkan data pengujian tak berlabel untuk membantu model mengenali distribusi data pengujian dengan lebih baik.
- Terlihat pada perbandingan di atas, dengan menggunakan PseudoLabelling, model XGBoost mampu memperoleh estimasi RMSE yang lebih baik (lebih rendah) daripada XGBoost biasa.